<a href="https://colab.research.google.com/github/anmolpatel2805/Cybersecurity-Journey/blob/main/Severity_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Testing the Email Severity Model

This section demonstrates how to use the saved model to predict the severity of a new, user-provided email text.

In [1]:
import pickle
import re
import pandas as pd
from scipy.sparse import hstack
import pandas as pd


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Load the saved model bundle
with open("/content/drive/MyDrive/NLP_based_severity/models/email_severity_rf_model (5L).pkl", "rb") as f:
    bundle = pickle.load(f)

tfidf = bundle["vectorizer"]
rf_model = bundle["model"]
severity_labels = bundle["severity_labels"]
urgency_words = bundle["urgency_words"] # Load urgency_words from the bundle
phishing_targets = bundle["phishing_targets"] # Load phishing_targets from the bundle

# Define the preprocessing functions (re-using the ones from earlier)
def extract_body_enron(message):
    if not isinstance(message, str):
        return ""

    parts = re.split(r"\n\s*\n", message, maxsplit=1)
    body = parts[1] if len(parts) > 1 else message

    reply_patterns = [
        r"\nFrom:.*",
        r"\nTo:.*",
        r"\nCc:.*",
        r"\nSubject:.*",
        r"\n-----Original Message-----",
        r"\nOn .* wrote:.*"
    ]

    for pattern in reply_patterns:
        body = re.split(pattern, body, maxsplit=1, flags=re.IGNORECASE)[0]

    body = re.sub(r"^[=\-_*]{3,}$", "", body, flags=re.MULTILINE)
    body = re.sub(r"\n{2,}", "\n", body)

    return body.strip()

def process_urls(text):
    if not isinstance(text, str):
        return "", 0, ""

    url_pattern = r'(https?://\S+|www\.\S+)'
    urls = re.findall(url_pattern, text)
    urls_str = ", ".join(urls) if len(urls) > 0 else ""
    url_count = len(urls)
    clean_text = re.sub(url_pattern, '', text)

    return urls_str, url_count, clean_text.strip()

def clean_text_nlp(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " url ", text)  # keep URL signal
    text = re.sub(r"[^a-z0-9\s]", " ", text)        # keep numbers
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def match_targets(email_text, target_list):
    if pd.isna(email_text):
        return []
    email_text = email_text.lower()
    email_words = set(re.findall(r'[a-z]+', email_text))
    return list(email_words.intersection(target_list))

def predict_email_severity(raw_email_text):
    # 1. Extract body
    body = extract_body_enron(raw_email_text)

    # 2. Process URLs
    urls_str, url_count, clean_body = process_urls(body)

    # 3. Clean text for NLP
    final_text = clean_text_nlp(clean_body)

    # 4. Calculate urgency count
    urgency_count = 0
    if pd.notna(final_text):
        text_lower = final_text.lower()
        for w in urgency_words: # Use loaded urgency_words
            if w in text_lower:
                urgency_count += 1

    # 5. Calculate target match count
    matched_targets = match_targets(final_text, phishing_targets) # Use loaded phishing_targets
    target_match_count = len(matched_targets)

    # 6. Vectorize text features
    X_text_features = tfidf.transform([final_text]) # Pass as list because transform expects iterable

    # 7. Combine all features
    extra_features = pd.DataFrame([{'urgency_count': urgency_count, 'target_match_count': target_match_count}])
    X_combined = hstack([X_text_features, extra_features])

    # 8. Predict severity
    predicted_severity_idx = rf_model.predict(X_combined)[0]
    # The model predicts string labels directly, so no need for index lookup
    return predicted_severity_idx

print("Model and preprocessing functions loaded and ready for prediction.")

Model and preprocessing functions loaded and ready for prediction.


In [4]:
pd.set_option('display.max_colwidth', None)

email_texts = [
    "Dear Investor,Greetings from Kotak Mahindra Mutual Fund. In pursuance to SEBI circular no. SEBI/HO/IMD/IMD-PoD-1/P/CIR/2024/90 dated June 27, 2024, the disclosure of Half-yearly Portfolio for the schemes of Kotak Mahindra Mutual Fund for the half year ended March 31, 2026 is available on www.kotakmf.com. Alternatively, you may also click on the link(s) provided below to access the portfolio of your invested scheme(s)",
    "Subject\n\nDear Team,\n\nI hope you are doing well.\n\nI would like to bring to your attention an issue that has been affecting our current workflow. The problem has started to impact productivity and may lead to further complications if not addressed promptly.\n\nI request you to kindly look into this matter at the earliest convenience and provide an update on the resolution timeline. Please let me know if any additional information is required from my end.\n\nThank you for your support.\n\nBest regards,\n[Your Name]",
    "SUbject\nMeeting Update,guys today meeting time will be share soon"
]

predictions = []
for email in email_texts:
    predicted_severity = predict_email_severity(email)
    predictions.append(predicted_severity)

output_df = pd.DataFrame({
    'Email Text': email_texts,
    'Predicted Severity': predictions
})

print("\n")
display(output_df)
print("\n")

,Email Text,Predicted Severity
0,"Dear Investor,Greetings from Kotak Mahindra Mutual Fund. In pursuance to SEBI circular no. SEBI/HO/IMD/IMD-PoD-1/P/CIR/2024/90 dated June 27, 2024, the disclosure of Half-yearly Portfolio for the schemes of Kotak Mahindra Mutual Fund for the half year ended March 31, 2026 is available on www.kotakmf.com. Alternatively, you may also click on the link(s) provided below to access the portfolio of your invested scheme(s)",Medium
1,"Subject\n\nDear Team,\n\nI hope you are doing well.\n\nI would like to bring to your attention an issue that has been affecting our current workflow. The problem has started to impact productivity and may lead to further complications if not addressed promptly.\n\nI request you to kindly look into this matter at the earliest convenience and provide an update on the resolution timeline. Please let me know if any additional information is required from my end.\n\nThank you for your support.\n\nBest regards,\n[Your Name]",Medium
2,"SUbject\nMeeting Update,guys today meeting time will be share soon",Low


### Explanation of Email Severity Predictions

Based on the model's logic and the content of each email, here's why the predictions were made:

1.  **Email 1 (Predicted Severity: High)**:
    *   **Reasoning**: This email contains numerous strong urgency indicators and phishing-related characteristics. Key phrases like "Urgent Account Suspension," "Immediate action is required," "Verify and confirm your account now," and "Failure to act immediately may result in permanent account closure" trigger the `urgency_count` feature. The presence of a suspicious-looking URL (`http://cliente.fidelidade-cielo.kinghost.net/cadastro.php`) likely contributed to a higher `target_match_count` due to domain names or keywords within the URL. The overall threatening and demanding tone, combined with a request for immediate action via an external link, are strong indicators of a high-severity, potentially malicious, email.

2.  **Email 2 (Predicted Severity: Medium)**:
    *   **Reasoning**: This email indicates a business-related issue that needs attention, but without the immediate threat or malicious intent of the first email. Phrases such as "affecting our current workflow," "impact productivity," and "addressed promptly" contribute to a moderate `urgency_count`. While it requests prompt action and an update, it lacks the critical, security-related keywords or suspicious links that would escalate its severity to 'High'. The context is clearly internal and problem-solving, making it of medium importance.

3.  **Email 3 (Predicted Severity: Low)**:
    *   **Reasoning**: " It contains no strong urgency keywords from our defined `urgency_words` list and no phishing indicators. The tone is casual and non-demanding, placing it in the low-severity category. The request is informational and not time-sensitive or critical.

In [5]:
pd.set_option('display.max_colwidth', None)

sample_email_input = input("Enter the email text you want to predict severity for: ")
predicted_severity = predict_email_severity(sample_email_input)

output_df = pd.DataFrame({
    'Email Text': [sample_email_input],
    'Predicted Severity': [predicted_severity]
})

print("\n")
display(output_df)
print("\n")

Enter the email text you want to predict severity for: hiii




,Email Text,Predicted Severity
0,hiii,Low
